# BEH 50: commercial outcomes

What happened after each customer month-end (`beh.CustomerMonthOutcome`), kept apart from the behaviour readings:
every outcome window starts the day after the month-end (clarifications D4).

- **Retention:** the ods_v2.PolicyRenewal offered / retained flags as they stand (D1). Policy level on
  `beh.CustomerPolicyMonth`; here the customer's nearest upcoming Motor renewal.
- **Cross-sell (D2):** a new-business Motor, Home or Van policy bought within 3, 6 or 12 months for a product the
  customer did not hold at the month-end. Customers who did not buy are kept (0); a product already held is not
  eligible (null). Rebooked mid-term changes and reversals (SaleCount <= 0) are excluded. Travel, Life and KeyCare
  are out of scope for now.
- **Cost to serve, track 1 (D3):** service effort in the next 3 and 12 months (calls, talk time, contacts, MTAs,
  document requests, chases, failed payments, friction, arrears). Track 2, the actual renewal processing channel,
  is RenewalSubChannel on beh.CustomerPolicyMonth and NextMotorRenewalSubChannel here; Finance's channel averages
  are applied outside the dataset.

An outcome whose window ends after the data end is null, with its *Observable* flag 0.

**If a cell fails**

1. Read the failing cell's header: what it builds, what it reads, its grain and key.
2. Fix the cause (usually a renamed or missing source column, or a broken key), then rerun from that cell.
   Every cell is `create or replace`, so rerunning is safe.
3. The checks cell at the end names any table that is empty or has duplicate / null keys, and the likely cause.

Run order: BEH_00_Setup is run by every notebook. Then 10, 20, 30, 40, 50, 60, in that order.


In [ ]:
%%pyspark
# Parameters (leave blank when running by hand)
batch_id = ""


In [ ]:
%%pyspark
%run BEH_00_Setup

In [ ]:
%%pyspark
beh_run_log("BEH_50_Outcomes", "started")


## beh_nb_sale


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_nb_sale                                                                   */
/*  Description : New-business Motor, Home and Van sales, first sale per policy code, linked to     */
/*                the customer. Rebooked mid-term changes (MTARebookedFlag Y) and reversals or      */
/*                zero rows (SaleCount <= 0) excluded.                                              */
/*  Reads       : ods_v2.PolicySale, stg.beh_policy_scv                                             */
/*  Key         : PolicyCode                                                                        */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_nb_sale as
select  l.SCV_Customer_Key,
        s.PolicyCode,
        min(s.PolicyTypeGroup)      as PolicyTypeGroup,
        min(s.ReportingSaleDate)    as SaleDate
from    ${beh.ods}.policysale s
join    ${beh.stg}.beh_policy_scv l on l.PolicyCode = s.PolicyCode
where   s.ReportingSaleType = 'New Business'
  and   s.PolicyTypeGroup in ('Motor', 'Home', 'Van')
  and   s.SaleCount > 0
  and   coalesce(s.MTARebookedFlag, 'N') <> 'Y'
group by l.SCV_Customer_Key, s.PolicyCode


## beh_xsell


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_xsell                                                                     */
/*  Description : Per customer month-end: whether a product not held at the month-end was bought    */
/*                within 3, 6 and 12 months, and the first such purchase in 12 months. Null = not   */
/*                eligible (product already held).                                                  */
/*  Reads       : beh.CustomerMonth, stg.beh_nb_sale                                                */
/*  Key         : SCV_Customer_Key + MonthEndDate                                                   */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_xsell as
select  c.SCV_Customer_Key,
        c.MonthEndDate,
        case when c.HoldsMotor = 0 then max(case when s.PolicyTypeGroup = 'Motor' and s.SaleDate <= add_months(c.MonthEndDate, 3) then 1 else 0 end) end as XSellMotor3m,
        case when c.HoldsHome = 0 then max(case when s.PolicyTypeGroup = 'Home' and s.SaleDate <= add_months(c.MonthEndDate, 3) then 1 else 0 end) end as XSellHome3m,
        case when c.HoldsVan = 0 then max(case when s.PolicyTypeGroup = 'Van' and s.SaleDate <= add_months(c.MonthEndDate, 3) then 1 else 0 end) end as XSellVan3m,
        case when c.ProductsHeld < 3 then max(case when ((s.PolicyTypeGroup = 'Motor' and c.HoldsMotor = 0) or (s.PolicyTypeGroup = 'Home' and c.HoldsHome = 0) or (s.PolicyTypeGroup = 'Van' and c.HoldsVan = 0)) and s.SaleDate <= add_months(c.MonthEndDate, 3) then 1 else 0 end) end as XSellAny3m,
        case when c.HoldsMotor = 0 then max(case when s.PolicyTypeGroup = 'Motor' and s.SaleDate <= add_months(c.MonthEndDate, 6) then 1 else 0 end) end as XSellMotor6m,
        case when c.HoldsHome = 0 then max(case when s.PolicyTypeGroup = 'Home' and s.SaleDate <= add_months(c.MonthEndDate, 6) then 1 else 0 end) end as XSellHome6m,
        case when c.HoldsVan = 0 then max(case when s.PolicyTypeGroup = 'Van' and s.SaleDate <= add_months(c.MonthEndDate, 6) then 1 else 0 end) end as XSellVan6m,
        case when c.ProductsHeld < 3 then max(case when ((s.PolicyTypeGroup = 'Motor' and c.HoldsMotor = 0) or (s.PolicyTypeGroup = 'Home' and c.HoldsHome = 0) or (s.PolicyTypeGroup = 'Van' and c.HoldsVan = 0)) and s.SaleDate <= add_months(c.MonthEndDate, 6) then 1 else 0 end) end as XSellAny6m,
        case when c.HoldsMotor = 0 then max(case when s.PolicyTypeGroup = 'Motor' and s.SaleDate <= add_months(c.MonthEndDate, 12) then 1 else 0 end) end as XSellMotor12m,
        case when c.HoldsHome = 0 then max(case when s.PolicyTypeGroup = 'Home' and s.SaleDate <= add_months(c.MonthEndDate, 12) then 1 else 0 end) end as XSellHome12m,
        case when c.HoldsVan = 0 then max(case when s.PolicyTypeGroup = 'Van' and s.SaleDate <= add_months(c.MonthEndDate, 12) then 1 else 0 end) end as XSellVan12m,
        case when c.ProductsHeld < 3 then max(case when ((s.PolicyTypeGroup = 'Motor' and c.HoldsMotor = 0) or (s.PolicyTypeGroup = 'Home' and c.HoldsHome = 0) or (s.PolicyTypeGroup = 'Van' and c.HoldsVan = 0)) and s.SaleDate <= add_months(c.MonthEndDate, 12) then 1 else 0 end) end as XSellAny12m,
        min(case when (s.PolicyTypeGroup = 'Motor' and c.HoldsMotor = 0) or (s.PolicyTypeGroup = 'Home' and c.HoldsHome = 0) or (s.PolicyTypeGroup = 'Van' and c.HoldsVan = 0) then s.SaleDate end) as FirstXSellDate12m,
        min_by(case when (s.PolicyTypeGroup = 'Motor' and c.HoldsMotor = 0) or (s.PolicyTypeGroup = 'Home' and c.HoldsHome = 0) or (s.PolicyTypeGroup = 'Van' and c.HoldsVan = 0) then s.PolicyTypeGroup end, case when (s.PolicyTypeGroup = 'Motor' and c.HoldsMotor = 0) or (s.PolicyTypeGroup = 'Home' and c.HoldsHome = 0) or (s.PolicyTypeGroup = 'Van' and c.HoldsVan = 0) then s.SaleDate end) as FirstXSellProduct12m
from    ${beh.target}.CustomerMonth c
left join ${beh.stg}.beh_nb_sale s
        on  s.SCV_Customer_Key = c.SCV_Customer_Key
        and s.SaleDate > c.MonthEndDate
        and s.SaleDate <= add_months(c.MonthEndDate, 12)
group by c.SCV_Customer_Key, c.MonthEndDate, c.HoldsMotor, c.HoldsHome, c.HoldsVan, c.ProductsHeld


## beh_next_motor_renewal


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_next_motor_renewal                                                        */
/*  Description : Per customer month-end, the Motor policy with the nearest upcoming renewal and    */
/*                that renewal's outcome (ties: lowest policy code).                                */
/*  Reads       : beh.CustomerPolicyMonth                                                           */
/*  Key         : SCV_Customer_Key + MonthEndDate                                                   */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_next_motor_renewal as
select  SCV_Customer_Key,
        MonthEndDate,
        RenewalDate             as NextMotorRenewalDate,
        RenewalOffered,
        RenewalRetained,
        RenewalOutcomeObservable,
        RenewalSubChannel
from    (select  p.*,
                 row_number() over (partition by p.SCV_Customer_Key, p.MonthEndDate
                                    order by p.RenewalDate, p.PolicyCode) as rn
         from    ${beh.target}.CustomerPolicyMonth p
         where   p.PolicyTypeGroup = 'Motor') x
where   rn = 1


## CustomerMonthOutcome


In [ ]:
/****************************************************************************************************/
/*  Name        : beh.CustomerMonthOutcome                                                          */
/*  Description : Outcomes after each customer month-end: cross-sell within 3 / 6 / 12 months,      */
/*                service effort in the next 3 / 12 months (cost to serve, track 1) and the         */
/*                nearest Motor renewal's outcome and processing channel. Null where the window is  */
/*                not yet observable.                                                               */
/*  Reads       : beh.CustomerMonth, stg.beh_xsell, stg.beh_rolling (forward sums),                 */
/*                stg.beh_next_motor_renewal                                                        */
/*  Grain       : customer x month-end (as beh.CustomerMonth)                                       */
/*  Key         : SCV_Customer_Key + MonthEndDate                                                   */
/****************************************************************************************************/
create or replace table ${beh.target}.CustomerMonthOutcome as
    select  c.SCV_Customer_Key,
            c.MonthEndDate,
            case when add_months(c.MonthEndDate, 3) <= to_date('${beh.outcome_end}') then 1 else 0 end as XSellObservable3m,
case when add_months(c.MonthEndDate, 3) <= to_date('${beh.outcome_end}') then x.XSellMotor3m end as XSellMotor3m,
case when add_months(c.MonthEndDate, 3) <= to_date('${beh.outcome_end}') then x.XSellHome3m end as XSellHome3m,
case when add_months(c.MonthEndDate, 3) <= to_date('${beh.outcome_end}') then x.XSellVan3m end as XSellVan3m,
case when add_months(c.MonthEndDate, 3) <= to_date('${beh.outcome_end}') then x.XSellAny3m end as XSellAny3m,
case when add_months(c.MonthEndDate, 6) <= to_date('${beh.outcome_end}') then 1 else 0 end as XSellObservable6m,
case when add_months(c.MonthEndDate, 6) <= to_date('${beh.outcome_end}') then x.XSellMotor6m end as XSellMotor6m,
case when add_months(c.MonthEndDate, 6) <= to_date('${beh.outcome_end}') then x.XSellHome6m end as XSellHome6m,
case when add_months(c.MonthEndDate, 6) <= to_date('${beh.outcome_end}') then x.XSellVan6m end as XSellVan6m,
case when add_months(c.MonthEndDate, 6) <= to_date('${beh.outcome_end}') then x.XSellAny6m end as XSellAny6m,
case when add_months(c.MonthEndDate, 12) <= to_date('${beh.outcome_end}') then 1 else 0 end as XSellObservable12m,
case when add_months(c.MonthEndDate, 12) <= to_date('${beh.outcome_end}') then x.XSellMotor12m end as XSellMotor12m,
case when add_months(c.MonthEndDate, 12) <= to_date('${beh.outcome_end}') then x.XSellHome12m end as XSellHome12m,
case when add_months(c.MonthEndDate, 12) <= to_date('${beh.outcome_end}') then x.XSellVan12m end as XSellVan12m,
case when add_months(c.MonthEndDate, 12) <= to_date('${beh.outcome_end}') then x.XSellAny12m end as XSellAny12m,
case when add_months(c.MonthEndDate, 12) <= to_date('${beh.outcome_end}') then x.FirstXSellDate12m end as FirstXSellDate12m,
case when add_months(c.MonthEndDate, 12) <= to_date('${beh.outcome_end}') then x.FirstXSellProduct12m end as FirstXSellProduct12m,
case when add_months(c.MonthEndDate, 3) <= to_date('${beh.last_me}') then 1 else 0 end as C2SObservable3m,
case when add_months(c.MonthEndDate, 3) <= to_date('${beh.last_me}') and date_add(c.MonthEndDate, 1) >= to_date('${beh.cov_calls}') then coalesce(r.calls_in_next3m, 0) end as c2s_calls_in_next3m,
case when add_months(c.MonthEndDate, 3) <= to_date('${beh.last_me}') and date_add(c.MonthEndDate, 1) >= to_date('${beh.cov_calls}') then coalesce(r.calls_in_talk_s_next3m, 0) end as c2s_calls_in_talk_s_next3m,
case when add_months(c.MonthEndDate, 3) <= to_date('${beh.last_me}') and date_add(c.MonthEndDate, 1) >= to_date('${beh.cov_calls}') then coalesce(r.calls_out_next3m, 0) end as c2s_calls_out_next3m,
case when add_months(c.MonthEndDate, 3) <= to_date('${beh.last_me}') and date_add(c.MonthEndDate, 1) >= to_date('${beh.cov_calls}') then coalesce(r.calls_out_talk_s_next3m, 0) end as c2s_calls_out_talk_s_next3m,
case when add_months(c.MonthEndDate, 3) <= to_date('${beh.last_me}') and date_add(c.MonthEndDate, 1) >= to_date('${beh.cov_calls}') then coalesce(r.inlife_contacts_next3m, 0) end as c2s_inlife_contacts_next3m,
case when add_months(c.MonthEndDate, 3) <= to_date('${beh.last_me}') and date_add(c.MonthEndDate, 1) >= to_date('${beh.cov_mta_count}') then coalesce(r.mtas_next3m, 0) end as c2s_mtas_next3m,
case when add_months(c.MonthEndDate, 3) <= to_date('${beh.last_me}') and date_add(c.MonthEndDate, 1) >= to_date('${beh.cov_calls}') then coalesce(r.doc_requests_next3m, 0) end as c2s_doc_requests_next3m,
case when add_months(c.MonthEndDate, 3) <= to_date('${beh.last_me}') and date_add(c.MonthEndDate, 1) >= to_date('${beh.cov_mychill}') then coalesce(r.chases_next3m, 0) end as c2s_chases_next3m,
case when add_months(c.MonthEndDate, 3) <= to_date('${beh.last_me}') and date_add(c.MonthEndDate, 1) >= to_date('${beh.cov_events}') then coalesce(r.failed_payments_next3m, 0) end as c2s_failed_payments_next3m,
case when add_months(c.MonthEndDate, 3) <= to_date('${beh.last_me}') and date_add(c.MonthEndDate, 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_events_next3m, 0) end as c2s_friction_events_next3m,
case when add_months(c.MonthEndDate, 3) <= to_date('${beh.last_me}') and date_add(c.MonthEndDate, 1) >= to_date('${beh.cov_arrears}') then coalesce(r.arrears_events_next3m, 0) end as c2s_arrears_events_next3m,
case when add_months(c.MonthEndDate, 12) <= to_date('${beh.last_me}') then 1 else 0 end as C2SObservable12m,
case when add_months(c.MonthEndDate, 12) <= to_date('${beh.last_me}') and date_add(c.MonthEndDate, 1) >= to_date('${beh.cov_calls}') then coalesce(r.calls_in_next12m, 0) end as c2s_calls_in_next12m,
case when add_months(c.MonthEndDate, 12) <= to_date('${beh.last_me}') and date_add(c.MonthEndDate, 1) >= to_date('${beh.cov_calls}') then coalesce(r.calls_in_talk_s_next12m, 0) end as c2s_calls_in_talk_s_next12m,
case when add_months(c.MonthEndDate, 12) <= to_date('${beh.last_me}') and date_add(c.MonthEndDate, 1) >= to_date('${beh.cov_calls}') then coalesce(r.calls_out_next12m, 0) end as c2s_calls_out_next12m,
case when add_months(c.MonthEndDate, 12) <= to_date('${beh.last_me}') and date_add(c.MonthEndDate, 1) >= to_date('${beh.cov_calls}') then coalesce(r.calls_out_talk_s_next12m, 0) end as c2s_calls_out_talk_s_next12m,
case when add_months(c.MonthEndDate, 12) <= to_date('${beh.last_me}') and date_add(c.MonthEndDate, 1) >= to_date('${beh.cov_calls}') then coalesce(r.inlife_contacts_next12m, 0) end as c2s_inlife_contacts_next12m,
case when add_months(c.MonthEndDate, 12) <= to_date('${beh.last_me}') and date_add(c.MonthEndDate, 1) >= to_date('${beh.cov_mta_count}') then coalesce(r.mtas_next12m, 0) end as c2s_mtas_next12m,
case when add_months(c.MonthEndDate, 12) <= to_date('${beh.last_me}') and date_add(c.MonthEndDate, 1) >= to_date('${beh.cov_calls}') then coalesce(r.doc_requests_next12m, 0) end as c2s_doc_requests_next12m,
case when add_months(c.MonthEndDate, 12) <= to_date('${beh.last_me}') and date_add(c.MonthEndDate, 1) >= to_date('${beh.cov_mychill}') then coalesce(r.chases_next12m, 0) end as c2s_chases_next12m,
case when add_months(c.MonthEndDate, 12) <= to_date('${beh.last_me}') and date_add(c.MonthEndDate, 1) >= to_date('${beh.cov_events}') then coalesce(r.failed_payments_next12m, 0) end as c2s_failed_payments_next12m,
case when add_months(c.MonthEndDate, 12) <= to_date('${beh.last_me}') and date_add(c.MonthEndDate, 1) >= to_date('${beh.cov_calls}') then coalesce(r.friction_events_next12m, 0) end as c2s_friction_events_next12m,
case when add_months(c.MonthEndDate, 12) <= to_date('${beh.last_me}') and date_add(c.MonthEndDate, 1) >= to_date('${beh.cov_arrears}') then coalesce(r.arrears_events_next12m, 0) end as c2s_arrears_events_next12m,
n.NextMotorRenewalDate as OutcomeMotorRenewalDate,
n.RenewalOffered as NextMotorRenewalOffered,
case when n.RenewalOutcomeObservable = 1 then n.RenewalRetained end as NextMotorRenewalRetained,
n.RenewalOutcomeObservable as NextMotorRenewalObservable,
case when n.RenewalOutcomeObservable = 1 then n.RenewalSubChannel end as NextMotorRenewalSubChannel
    from    ${beh.target}.CustomerMonth c
    left join ${beh.stg}.beh_xsell x
            on  x.SCV_Customer_Key = c.SCV_Customer_Key
            and x.MonthEndDate = c.MonthEndDate
    left join ${beh.stg}.beh_rolling r
            on  r.SCV_Customer_Key = c.SCV_Customer_Key
            and r.MonthEndDate = c.MonthEndDate
    left join ${beh.stg}.beh_next_motor_renewal n
            on  n.SCV_Customer_Key = c.SCV_Customer_Key
            and n.MonthEndDate = c.MonthEndDate


## Checks


In [ ]:
%%pyspark
# Row count, duplicate keys and null keys for every table this notebook built. Logged to BehBuildLog;
# the cell fails if any check failed, naming the table and the likely cause.
beh_check("BEH_50_Outcomes", [{"table": "{stg}.beh_nb_sale", "key": ["PolicyCode"]}, {"table": "{stg}.beh_xsell", "key": ["SCV_Customer_Key", "MonthEndDate"]}, {"table": "{stg}.beh_next_motor_renewal", "key": ["SCV_Customer_Key", "MonthEndDate"]}, {"table": "{target}.CustomerMonthOutcome", "key": ["SCV_Customer_Key", "MonthEndDate"]}])


In [ ]:
%%pyspark
beh_run_log("BEH_50_Outcomes", "succeeded")
